# 🚀 Pipeline Orchestration - End-to-End

This notebook runs the complete Medallion Architecture pipeline:

```
🥉 Bronze → 🥈 Silver → 🥇 Gold
```

## Pipeline Flow

| Step | Layer | Description | LLM |
|------|------|-------------|-----|
| 1 | Bronze | Raw data ingestion from 5 sources | None |
| 2 | Silver | Spatial standardization + ALL AI enrichment | **YES** |
| 3 | Gold | Kimball Star Schema modeling | **NO** |

## Shift-Left AI Pattern
- **ALL LLM inference happens in Silver layer**
- **Gold layer is purely declarative SQL** (no AI calls)

In [ ]:
# Import all layer modules
import sys
sys.path.append('../src')

from bronze_ingestion import Config as BronzeConfig, run_bronze_ingestion
from silver_enrichment import (
    Config as SilverConfig,
    create_spark_session,
    create_ollama_enrichment_udf,
    parse_ai_enrichment,
    write_silver_table
)
from gold_dimensional_modeling import (
    Config as GoldConfig,
    create_spark_session as create_gold_spark,
    create_dim_neighborhoods,
    create_dim_infrastructure,
    create_fact_hazard_events,
    write_gold_table
)

print("✅ All modules imported")

## Configuration Check

In [ ]:
# Show all configurations

bronze_cfg = BronzeConfig()

silver_cfg = SilverConfig()

gold_cfg = GoldConfig()



print("=" * 60)

print("PIPELINE CONFIGURATION")

print("=" * 60)

print()

print("🥉 BRONZE:")

print(f"   Data Dir: {bronze_cfg.LOCAL_DATA_DIR}")

print(f"   Bucket: {bronze_cfg.BRONZE_BUCKET}")

print()

print("🥈 SILVER:")

print(f"   Ollama URL: {silver_cfg.OLLAMA_BASE_URL}")

print(f"   Ollama Model: {silver_cfg.OLLAMA_MODEL}")

print()

print("🥇 GOLD:")

print(f"   Output: {gold_cfg.GOLD_BUCKET}")

## Step 1: Run Bronze Layer

In [ ]:
# Run Bronze ingestion
print("=" * 60)
print("STEP 1: BRONZE LAYER")
print("=" * 60)

bronze_success = run_bronze_ingestion()
print(f"\n{'✅ SUCCESS' if bronze_success else '❌ FAILED'}: Bronze ingestion")

## Step 2: Run Silver Layer

In [ ]:
# Run Silver layer (using local Spark)
print("=" * 60)
print("STEP 2: SILVER LAYER")  
print("=" * 60)

# Create Spark session
spark = create_spark_session(silver_cfg)
print(f"✅ Spark session: {spark.version}")

# Demonstrate Ollama UDF
print("\n📝 Testing Ollama enrichment function:")
result = parse_ai_enrichment('{"severity": 7, "hazard_type": "traffic"}')
print(f"   Parsed: {result}")

# Note: Full Silver pipeline requires running Ollama container
# For demo, we verify the functions are available
print("\n✅ Silver layer ready for execution")

## Step 3: Run Gold Layer

In [ ]:
# Run Gold layer (using local Spark)
print("=" * 60)
print("STEP 3: GOLD LAYER")  
print("=" * 60)

# Create Spark session
gold_spark = create_gold_spark(gold_cfg)
print(f"✅ Spark session: {gold_spark.version}")

# Create dimension tables
dim_neighborhoods = create_dim_neighborhoods(gold_spark)
dim_infrastructure = create_dim_infrastructure(gold_spark)
fact_hazard = create_fact_hazard_events(gold_spark)

print(f"\n✅ Created tables:")
print(f"   - dim_neighborhoods: {dim_neighborhoods.count()} rows")
print(f"   - dim_infrastructure: {dim_infrastructure.count()} rows")
print(f"   - fact_hazard_events: {fact_hazard.count()} rows")

## Pipeline Results

In [ ]:
# Display final summary
print()
print("╔" + "═" * 58 + "╗")
print("║" + " ".center(58) + "║")
print("║" + "PIPELINE COMPLETE".center(58) + "║")
print("║" + " ".center(58) + "║")
print("╚" + "═" * 58 + "╝")
print()
print("📊 Pipeline Summary:")
print("-" * 60)
print("Layer    │ Output Location           │ LLM    │")
print("-" * 60)
print("Bronze  │ /tmp/geoai/bronze/      │ None   │")
print("Silver  │ /tmp/geoai/silver/      │ ALL    │")
print("Gold    │ /tmp/geoai/gold/        │ NONE   │")
print("-" * 60)
print()
print("🎯 Shift-Left AI Pattern:")
print("  ✅ ALL LLM inference in Silver layer")
print("  ✅ Gold layer uses pure SQL transformations")

## Verification SQL Queries

In [ ]:
# Example analytical queries (for verification)
from pyspark.sql import functions as F

print("📊 Example Analytics on Gold Layer:")
print()

# Query 1: Events by neighborhood
print("1. Events per neighborhood:")
if fact_hazard.count() > 0:
    fact_hazard.groupBy("neighborhood_id").count().show()
else:
    print("   (No data - would show count by neighborhood)")

# Query 2: Average severity by event source
print("2. Average severity by source:")
if fact_hazard.count() > 0:
    fact_hazard.groupBy("source").agg(
        F.avg("severity").alias("avg_severity")
    ).show()
else:
    print("   (No data - would show avg severity)")

---

## Next Steps

### Individual Notebooks
- `01_bronze_ingestion.ipynb` - Data ingestion
- `02_silver_enrichment.ipynb` - Spatial + AI
- `03_gold_dimensional.ipynb` - Star Schema

### Testing
```bash
pytest tests/test_silver_enrichment.py -v
```

### Docker Deployment
```bash
docker-compose up -d
```